# Populate the population table

Run `populate_area.ipynb` first. This notebook loads **mid-2021** population and published median age for every 2021 LSOA already in `area`, into the same `data/processed/database.db`. Paths work from the repository root or `data/`.

The six attributes follow the existing snake_case convention: `id`, `area_id` (your areaID), `year`, `population` (number of usual residents), `median_income` (individual annual income, GBP), and `median_age` (years). A unique `(area_id, year)` key makes re-runs update rows while keeping their IDs.

**Income gap:** no compatible 2021 LSOA individual median-income series was verified. The default import therefore stores SQL NULL, never zero, for missing income. The optional import below accepts only documented, same-year medians on 2021 LSOA boundaries. It does not mislabel older income as 2021 data.


## Sources and definitions

- **Population:** [ONS LSOA population estimates](https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/lowersuperoutputareamidyearpopulationestimates), edition *Mid-2021 and mid-2022*, sheet *Mid-2021 LSOA 2021*, published Total. Reference date: 30 June 2021; all usual residents, both sexes. This is a mid-year estimate, not the March Census count. The download is about 53 MB.
- **Median age:** [ONS request 2093](https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/adhocs/2093estimatesofmedianageforlowerlayersuperoutputareasinenglandandwalesmid2021andmid2022), released 23 May 2024, sheet *Mid-2021 & 2022 LSOA*, column *median age 2021*. Published estimate, not an average of neighbourhood medians or an approximation from age bands. ONS supplies these ages as supporting information based on single-year age estimates.
- **Individual income candidate:** [ONS admin-based income data for individuals](https://www.ons.gov.uk/peoplepopulationandcommunity/personalandhouseholdfinances/incomeandwealth/datasets/adminbasedincomestatisticsdataforindividualsenglandandwales), tax year ending 2018, released 20 December 2022. Gross and net individual income measures from tax and benefits data; inspect the workbook's median/50th-percentile, population coverage and boundary definitions before use. This is a historical source, not a 2021 series. Changed LSOAs need boundary assessment; medians cannot be summed or population-weighted to reconstruct merged/split-area medians.
- [ABIS methodology](https://www.ons.gov.uk/peoplepopulationandcommunity/personalandhouseholdfinances/incomeandwealth/methodologies/adminbasedincomestatisticsqmi): Experimental Statistics with incomplete income components and population coverage; unsuitable for inferring current financial trends. Any later use must retain the experimental label and source qualifications.
- [ONS standard small-area income estimates](https://www.ons.gov.uk/peoplepopulationandcommunity/personalandhouseholdfinances/incomeandwealth/bulletins/smallareamodelbasedincomeestimates/financialyearending2023) estimate **mean household** income at MSOA level. They do not provide the requested individual median at LSOA level.

Both demographic downloads use Census 2021 LSOAs. Only 2021 is loaded: the older workbook's 2022 estimates were subsequently revised, so its 2022 values must not be imported as current estimates. Store unrounded counts for calculations; follow ONS guidance to round population estimates to the nearest hundred in published outputs. Sources are available under the Open Government Licence, subject to their notes.


## 1. Dependencies and settings

Use a Python 3 Jupyter kernel with pandas and openpyxl. If needed, run `%pip install pandas openpyxl` in a separate cell. No database changes occur until section 6.


In [ ]:
import hashlib
import json
import math
import sqlite3
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import pandas as pd
from openpyxl import load_workbook

cwd = Path.cwd()
DATA_DIR = cwd / "data" if (cwd / "data" / "populate_area.ipynb").is_file() else cwd
if not (DATA_DIR / "populate_area.ipynb").is_file():
    raise RuntimeError("Run from the repository root or its data directory.")
DB_PATH = DATA_DIR / "processed" / "database.db"
RAW_DIR = DATA_DIR / "raw" / "official" / "population"
YEAR = 2021  # pinned to the verified mid-2021 release and columns
INCOME_CSV = None  # Optional: Path to the documented CSV described in section 4.
INCOME_DEFINITION = None  # Required with CSV: describe gross/net, coverage and year basis.
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Run populate_area.ipynb first: {DB_PATH}")
with closing(sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)) as con:
    areas = pd.read_sql_query("SELECT id AS area_id, lsoa_code, name, lad_name FROM area", con)
if areas.empty or not areas["lsoa_code"].is_unique:
    raise ValueError("area must contain unique 2021 LSOA codes. Run populate_area.ipynb.")
print(f"{len(areas)} target LSOAs; database: {DB_PATH}")


## 2. Download and cache the two official workbooks

Existing cached files are reused. Each file gets a URL, retrieval timestamp and SHA-256 sidecar; invalid downloads are rejected before replacing a cache file. For offline use, place the official workbooks at these cache paths first.


In [ ]:
BASE = "https://www.ons.gov.uk/file?uri="
POP_URL = BASE + "/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/lowersuperoutputareamidyearpopulationestimates/mid2021andmid2022/sapelsoasyoatablefinal.xlsx"
AGE_URL = BASE + "/peoplepopulationandcommunity/populationandmigration/populationestimates/adhocs/2093estimatesofmedianageforlowerlayersuperoutputareasinenglandandwalesmid2021andmid2022/lsoamedian2021and2022.xlsx"

def cached_workbook(url, filename):
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    path = RAW_DIR / filename
    retrieved_at = None
    if not path.exists():
        print(f"Downloading {filename} ...")
        request = Request(url, headers={"User-Agent": "SouthamptonPopulationResearch/1.0"})
        with urlopen(request, timeout=180) as response:
            content = response.read()
        # Validate the response as an Excel workbook before saving.
        from io import BytesIO
        wb = load_workbook(BytesIO(content), read_only=True, data_only=True)
        wb.close()
        path.write_bytes(content)
        retrieved_at = datetime.now(timezone.utc).isoformat()
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    metadata_path = path.with_suffix(".source.json")
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata["sha256"] != digest or metadata["url"] != url:
            raise ValueError(f"Cached file or source changed: {path}. Inspect before reimporting.")
    else:
        metadata = {"url": url, "sha256": digest, "retrieved_at_utc": retrieved_at,
                    "note": "Retrieval date unknown for manually supplied cache." if retrieved_at is None else ""}
        metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return path

pop_path = cached_workbook(POP_URL, "sape_mid2021_mid2022.xlsx")
age_path = cached_workbook(AGE_URL, "lsoa_median_age_2021_2022.xlsx")


## 3. Read the verified LSOA columns

Stream just the required sheet and columns. Only LSOAs already in the database are retained, and every target must appear exactly once in each source.


In [ ]:
def select_lsoas(path, sheet, code_column, value_column, output_column):
    wb = load_workbook(path, read_only=True, data_only=True)
    records = []
    target_codes = set(areas["lsoa_code"])
    try:
        ws = wb[sheet]
        stream = ws.iter_rows(values_only=True)
        # Both verified workbooks have their column headings on Excel row 4.
        for _ in range(3):
            next(stream)
        headers = [str(value).strip() if value is not None else "" for value in next(stream)]
        if headers.count(code_column) != 1 or headers.count(value_column) != 1:
            raise ValueError(f"Unexpected headers in {path.name}/{sheet}: {headers[:8]}")
        ci, vi = headers.index(code_column), headers.index(value_column)
        for row in stream:
            code = str(row[ci]).strip() if row[ci] is not None else ""
            if code in target_codes:
                records.append((code, row[vi]))
    finally:
        wb.close()
    frame = pd.DataFrame(records, columns=["lsoa_code", output_column])
    if not frame["lsoa_code"].is_unique:
        raise ValueError(f"Duplicate target codes in {path.name}")
    missing = target_codes - set(frame["lsoa_code"])
    if missing:
        raise ValueError(f"{path.name} is missing target LSOAs: {sorted(missing)}")
    frame[output_column] = pd.to_numeric(frame[output_column], errors="raise")
    return frame

population = select_lsoas(pop_path, "Mid-2021 LSOA 2021",
                         "LSOA 2021 Code", "Total", "population")
median_age = select_lsoas(age_path, "Mid-2021 & 2022 LSOA",
                         "LSOA21CD", "median age 2021", "median_age")
data = areas.merge(population, on="lsoa_code", validate="one_to_one").merge(
    median_age, on="lsoa_code", validate="one_to_one")
data["year"] = YEAR
data["median_income"] = float("nan")
if not data["population"].map(lambda v: pd.notna(v) and math.isfinite(v) and v >= 0 and v == int(v)).all():
    raise ValueError("Population must be a nonnegative whole-number count.")
if not data["median_age"].map(lambda v: pd.notna(v) and math.isfinite(v) and 0 <= v <= 120).all():
    raise ValueError("Missing or invalid median ages.")
print(f"Matched all {len(data)} target areas.")
data[["lsoa_code", "name", "year", "population", "median_age"]].head()


## 4. Optional individual annual median income

Default: leave `INCOME_CSV = None`; demographics still load. To provide a verified compatible source, set the path and `INCOME_DEFINITION` in section 1 and rerun. Use a CSV with columns:

`lsoa_code,year,median_income,geography_vintage,statistic,unit,income_basis,source_url`

Each row must declare `year=2021`, `geography_vintage=2021`, `statistic=median`, `unit=GBP/year` and `income_basis=individual`. Select one gross/net definition for the whole file and describe it, along with the age/income coverage and calendar/tax-year basis, in `INCOME_DEFINITION`. Use annual pounds, not thousands or weekly earnings. Keep suppressed values blank; zero is accepted only when actually observed. Partial LSOA coverage is reported and remains NULL elsewhere.

These declarations record your source assessment; they do not make an incompatible source compatible. The 2018 ABIS download cannot pass this 2021 import. Do not change its year or average medians across boundaries. A historical income series requires separate population/age sources for its own reference year and verified boundary compatibility.


In [ ]:
income_provenance = None
if INCOME_CSV is not None:
    if not INCOME_DEFINITION or not INCOME_DEFINITION.strip():
        raise ValueError("Describe the individual income measure in INCOME_DEFINITION.")
    income_path = Path(INCOME_CSV)
    inc = pd.read_csv(income_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    required = {"lsoa_code", "year", "median_income", "geography_vintage",
                "statistic", "unit", "income_basis", "source_url"}
    if not required <= set(inc.columns):
        raise ValueError(f"Income CSV missing: {required - set(inc.columns)}")
    inc = inc.apply(lambda column: column.str.strip())
    if inc.empty or not inc["lsoa_code"].str.fullmatch(r"[EW]01\d{6}").all():
        raise ValueError("Income CSV needs valid LSOA codes.")
    if not inc["lsoa_code"].is_unique:
        raise ValueError("Duplicate income LSOAs; select one consistent income measure/year.")
    expected = {"year": str(YEAR), "geography_vintage": "2021", "statistic": "median",
                "unit": "GBP/year", "income_basis": "individual"}
    for column, value in expected.items():
        if not inc[column].eq(value).all():
            raise ValueError(f"{column} must be {value!r} throughout the income file.")
    if not inc["source_url"].str.match(r"https?://\S+$").all():
        raise ValueError("Every income row needs a source URL.")
    inc["median_income"] = pd.to_numeric(inc["median_income"].replace("", float("nan")), errors="raise")
    nonmissing = inc["median_income"].dropna()
    if not nonmissing.map(lambda v: math.isfinite(v) and v >= 0).all():
        raise ValueError("Income must be finite, nonnegative annual GBP or blank.")
    selected = inc[inc["lsoa_code"].isin(data["lsoa_code"])]
    if selected.empty:
        raise ValueError("Income file has no matching target LSOAs.")
    data["median_income"] = data["lsoa_code"].map(inc.set_index("lsoa_code")["median_income"])
    income_provenance = {"file": str(income_path.resolve()),
                         "sha256": hashlib.sha256(income_path.read_bytes()).hexdigest(),
                         "definition": INCOME_DEFINITION,
                         "source_urls": sorted(set(selected["source_url"]))}
print(f"Income available for {data['median_income'].notna().sum()} of {len(data)} target areas.")


## 5. Review the prepared rows

All source validation finishes before the database is opened for writing. No ward-level median is calculated: a median of LSOA medians is not a ward median.


In [ ]:
if len(data) != len(areas) or data.duplicated(["area_id", "year"]).any():
    raise ValueError("Prepared rows must contain exactly one row per target area/year.")
data[["area_id", "lsoa_code", "name", "year", "population", "median_income", "median_age"]].head(10)


## 6. Create the table and populate the existing database

This cell writes to SQLite. The schema, load and verification share one transaction. Upserts preserve IDs and keep an existing income value if this run supplies no replacement. No rows from other years are removed. An incompatible pre-existing population schema causes an error rather than an automatic migration.


In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS population (
    id            INTEGER PRIMARY KEY,
    area_id       INTEGER NOT NULL REFERENCES area(id),
    year          INTEGER NOT NULL CHECK (typeof(year) = 'integer' AND year BETWEEN 1900 AND 2100),
    population    INTEGER NOT NULL CHECK (typeof(population) = 'integer' AND population >= 0),
    median_income REAL CHECK (median_income IS NULL OR median_income >= 0),
    median_age    REAL NOT NULL CHECK (median_age BETWEEN 0 AND 120),
    UNIQUE (area_id, year)
)
"""
UPSERT = """
INSERT INTO population (area_id, year, population, median_income, median_age)
VALUES (?, ?, ?, ?, ?)
ON CONFLICT(area_id, year) DO UPDATE SET
    population = excluded.population,
    median_income = COALESCE(excluded.median_income, population.median_income),
    median_age = excluded.median_age
"""
rows = [(int(r.area_id), int(r.year), int(r.population),
         None if pd.isna(r.median_income) else float(r.median_income), float(r.median_age))
        for r in data.itertuples(index=False)]

with closing(sqlite3.connect(DB_PATH, timeout=30)) as con:
    con.execute("PRAGMA foreign_keys = ON")
    try:
        con.execute("BEGIN IMMEDIATE")
        con.execute(SCHEMA)
        actual_columns = [r[1] for r in con.execute("PRAGMA table_info(population)")]
        if actual_columns != ["id", "area_id", "year", "population", "median_income", "median_age"]:
            raise ValueError(f"Existing population schema differs: {actual_columns}")
        before_ids = dict(con.execute("SELECT area_id, id FROM population WHERE year = ?", (YEAR,)))
        con.executemany(UPSERT, rows)
        result = pd.read_sql_query(
            """SELECT p.*, a.lsoa_code, a.name, a.ward_name
               FROM population p JOIN area a ON a.id = p.area_id
               WHERE p.year = ? ORDER BY a.lsoa_code""", con, params=(YEAR,))
        actual = {int(r.area_id): r for r in result.itertuples(index=False)}
        for area_id, year, count, income, age in rows:
            saved = actual[area_id]
            if saved.population != count or saved.median_age != age:
                raise RuntimeError(f"Stored demographics differ for area {area_id}")
            if income is not None and saved.median_income != income:
                raise RuntimeError(f"Stored income differs for area {area_id}")
            if area_id in before_ids and saved.id != before_ids[area_id]:
                raise RuntimeError("An existing population ID changed.")
        violations = con.execute("PRAGMA foreign_key_check(population)").fetchall()
        if violations:
            raise RuntimeError(f"Broken population foreign keys: {violations[:5]}")
        con.commit()
    except Exception:
        con.rollback()
        raise

# Keep source hashes and income definition with this import's record.
manifest = {"year": YEAR, "reference_date": "2021-06-30", "geography": "LSOA 2021",
            "loaded_at_utc": datetime.now(timezone.utc).isoformat(),
            "rows_prepared": len(rows),
            "population_source": json.loads(pop_path.with_suffix(".source.json").read_text(encoding="utf-8")),
            "median_age_source": json.loads(age_path.with_suffix(".source.json").read_text(encoding="utf-8")),
            "income_source": income_provenance,
            "missing_income_policy": "NULL on insert; preserve existing value on update"}
manifest_path = RAW_DIR / ("import_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Loaded and verified {len(rows)} area-year rows in {DB_PATH}")
print(f"Provenance: {manifest_path}")
print(f"{result['median_income'].isna().sum()} stored rows have no individual income estimate.")
result.head(10)


## 7. Summary

Ward totals below sum population only. For income/age analysis, query the LSOA rows directly or obtain distributions appropriate to the desired larger geography.


In [ ]:
result.groupby("ward_name", dropna=False).agg(
    areas=("area_id", "count"),
    people=("population", "sum"),
    areas_with_income=("median_income", "count"),
).sort_values("people", ascending=False)
